# 04 — GitHub feature store ile Application Domain ve SHAP

Bu sürüm, mevcut GitHub repo yapısıyla uyumludur ve sabit, bulunmayan bir
`data/CHEMBL206_Mordred2D_model_ready.csv` bağlantısına bağlı değildir.

Öncelikli gerçek GitHub feature store yolu:

```text
molfet_regression_outputs/01_mordred_feature_store/mordred_2d_features.csv
```

Notebook:

- GitHub repo ağacını tarar.
- Uygun Mordred feature store dosyasını otomatik bulur.
- Model bundle dosyasını GitHub veya Google Drive'da arar.
- Bundle bulunamazsa aynı veri üzerinde tekrar üretilebilir bir Random Forest modeli kurar.
- Mordred matrisindeki `NaN`, `+inf` ve `-inf` değerlerini güvenli biçimde işler.
- Application Domain, Williams plot ve SHAP çıktılarını Google Drive'a kaydeder.

## Workshop akışı

```text
GitHub repo ağacı
        ↓
Mordred feature store seçimi
        ↓
Bundle/model arama
        ↓
Feature temizleme ve train tabanlı imputasyon
        ↓
Model yükleme veya tek seferlik eğitim
        ↓
PCA-leverage Application Domain
        ↓
SHAP global ve lokal açıklamalar
        ↓
Google Drive çıktıları
```

> Hücreleri yukarıdan aşağıya sırayla çalıştırınız.  
> Aynı split, aynı model ve aynı SHAP hesabı farklı hücrelerde tekrar edilmez.

## Hücre 1 — Paket kontrolü

In [ ]:
import importlib.util
# Paketlerin aktif Python ortamında kurulu olup olmadığını kontrol etmek için kullanılır.

import subprocess
# Eksik paketleri aktif Python ortamına kurmak için kullanılır.

import sys
# Colab oturumunda kullanılan aktif Python yorumlayıcısını belirlemek için kullanılır.


REQUIRED_PACKAGES = [
    ("numpy", "numpy"),
    # Sayısal matris ve leverage işlemleri için kullanılır.

    ("pandas", "pandas"),
    # CSV okuma ve sonuç tabloları için kullanılır.

    ("requests", "requests"),
    # GitHub API ve RAW bağlantılarına erişmek için kullanılır.

    ("matplotlib", "matplotlib"),
    # Williams ve SHAP grafiklerini kaydetmek için kullanılır.

    ("sklearn", "scikit-learn"),
    # Split, imputasyon, PCA ve modelleme için kullanılır.

    ("joblib", "joblib"),
    # Eğitilmiş modeli yüklemek ve kaydetmek için kullanılır.

    ("shap", "shap"),
    # Global ve lokal model açıklamaları için kullanılır.
]
# Gerekli paketlerin import ve pip adları tanımlanır.


for import_name, pip_name in REQUIRED_PACKAGES:
    # Paketler sırayla kontrol edilir.

    if importlib.util.find_spec(import_name) is None:
        # Paket aktif ortamda yoksa kurulum yapılır.

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                pip_name,
            ]
        )
        # Paket notebookun kullandığı aktif Python ortamına kurulur.


print("✅ CHECKPOINT 04.1: Paket kontrolü tamamlandı.")

## Hücre 2 — Kütüphanelerin içe aktarılması

In [ ]:
from pathlib import Path
# Google Drive ve çıktı dosya yollarını yönetmek için kullanılır.

from io import BytesIO
# GitHub'dan indirilen CSV içeriğini bellekte pandas'a aktarmak için kullanılır.

import json
# Model bundle ve analiz özetini JSON olarak okumak/yazmak için kullanılır.

import warnings
# Uzun model uyarılarını azaltmak için kullanılır.

warnings.filterwarnings("ignore")
# Workshop ekranını kalabalıklaştıran genel uyarılar gizlenir.

import joblib
# Eğitilmiş modeli yüklemek ve Drive'a kaydetmek için kullanılır.

import matplotlib.pyplot as plt
# Williams ve SHAP grafiklerini kaydetmek için kullanılır.

import numpy as np
# Leverage, residual ve SHAP matris işlemleri için kullanılır.

import pandas as pd
# GitHub CSV okuma ve AD tablolarını düzenlemek için kullanılır.

import requests
# GitHub API ve RAW dosya bağlantılarına istek göndermek için kullanılır.

import shap
# Global ve lokal model açıklamalarını hesaplamak için kullanılır.

from IPython.display import display
# Tabloları Colab içinde okunabilir biçimde göstermek için kullanılır.

from google.colab import drive
# Analiz çıktılarını Google Drive'a kaydetmek için kullanılır.

from sklearn.decomposition import PCA
# Descriptor uzayını leverage hesabı için PCA uzayına indirgemek için kullanılır.

from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
# Bundle içindeki veya fallback olarak seçilen ağaç modellerini oluşturmak için kullanılır.

from sklearn.impute import SimpleImputer
# Eksik descriptor değerlerini yalnızca eğitim medyanıyla doldurmak için kullanılır.

from sklearn.model_selection import train_test_split
# Bundle yoksa tekrar üretilebilir train/test ayrımı oluşturmak için kullanılır.

from sklearn.preprocessing import StandardScaler
# PCA öncesinde featureları yalnızca eğitim setinde standardize etmek için kullanılır.


print("Pandas sürümü:", pd.__version__)
print("SHAP sürümü:", shap.__version__)
print("✅ CHECKPOINT 04.2: Kütüphaneler başarıyla içe aktarıldı.")

## Hücre 3 — Google Drive bağlantısı

In [ ]:
drive.mount(
    "/content/drive",
    force_remount=False,
)
# Google Drive standart Colab dizinine bağlanır.


print("✅ CHECKPOINT 04.3: Google Drive bağlantısı hazır.")

## Hücre 4 — Target, GitHub ve analiz ayarları

In [ ]:
TARGET_ID = "CHEMBL206"
# İşlenecek ChEMBL Target ID burada değiştirilir.


GITHUB_OWNER = "MOL-FEST"
# GitHub organizasyon adı tanımlanır.


GITHUB_REPOSITORY = "MOL-FET_regression"
# GitHub repo adı tanımlanır.


GITHUB_BRANCH = "main"
# Dosyaların okunacağı GitHub branch adı tanımlanır.


TARGET_FEATURE_FILENAME = (
    f"{TARGET_ID}_Mordred2D_model_ready.csv"
)
# Workshop notebook 02 tarafından üretilebilecek target'a özel feature dosyası tanımlanır.


REPOSITORY_FEATURE_FILENAME = "mordred_2d_features.csv"
# Mevcut GitHub reposunda bulunan gerçek Mordred feature store adı tanımlanır.


BUNDLE_FILENAME = f"{TARGET_ID}_final_model_bundle.json"
# Önceki modelleme notebookunun üretebileceği model bundle adı tanımlanır.


MODEL_FILENAME = f"{TARGET_ID}_final_tree_model.joblib"
# Önceki modelleme notebookunun üretebileceği model dosya adı tanımlanır.


IMPUTER_FILENAME = f"{TARGET_ID}_feature_imputer.joblib"
# Önceki modelleme notebookunun üretebileceği imputer dosya adı tanımlanır.


DRIVE_ROOT = Path(
    "/content/drive/MyDrive/MOL_FET_regression_workshop"
)
# Bütün AD ve SHAP çıktılarının kaydedileceği Drive klasörü tanımlanır.


DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
# Drive klasörü yoksa oluşturulur.


RANDOM_STATE = 42
# Split ve fallback model için tekrar üretilebilir seed belirlenir.


TEST_SIZE = 0.20
# Bundle yoksa kullanılacak test oranı belirlenir.


MAX_FEATURE_MISSING_RATIO = 0.20
# Bir descriptorın kabul edilen maksimum eksik hücre oranı belirlenir.


MAX_SHAP_SAMPLES = 200
# Workshop süresini kontrol etmek için SHAP örneklem üst sınırı belirlenir.


REQUEST_TIMEOUT = 180
# GitHub isteklerinin zaman aşımı süresi saniye olarak belirlenir.


print("TARGET_ID:", TARGET_ID)
print("Drive çıktı klasörü:", DRIVE_ROOT)
print("✅ CHECKPOINT 04.4: Çalışma ayarları hazır.")

## Hücre 5 — GitHub aday dosya yolları

In [ ]:
FEATURE_GITHUB_PATHS = [
    f"data/{TARGET_FEATURE_FILENAME}",
    # Target'a özel dosyanın tercih edilen data/ klasörü yolu.

    TARGET_FEATURE_FILENAME,
    # Target'a özel dosya repo kökünde bulunursa kullanılacak yol.

    (
        "molfet_regression_outputs/"
        "01_mordred_feature_store/"
        "mordred_2d_features.csv"
    ),
    # Yüklenen gerçek GitHub reposunda bulunan Mordred feature store yolu.

    REPOSITORY_FEATURE_FILENAME,
    # Aynı feature store daha sonra repo köküne taşınırsa kullanılacak yol.
]
# Denenecek feature CSV yolları öncelik sırasıyla tanımlanır.


BUNDLE_GITHUB_PATHS = [
    f"data/{BUNDLE_FILENAME}",
    BUNDLE_FILENAME,
    f"molfet_regression_outputs/04_lazypredict_regression/{BUNDLE_FILENAME}",
]
# Model bundle için olası GitHub yolları tanımlanır.


MODEL_GITHUB_PATHS = [
    f"data/{MODEL_FILENAME}",
    MODEL_FILENAME,
    f"molfet_regression_outputs/04_lazypredict_regression/{MODEL_FILENAME}",
]
# Joblib model için olası GitHub yolları tanımlanır.


GITHUB_API_TREE_URL = (
    f"https://api.github.com/repos/{GITHUB_OWNER}/"
    f"{GITHUB_REPOSITORY}/git/trees/{GITHUB_BRANCH}?recursive=1"
)
# Repo içindeki bütün dosyaları recursive listeleyen GitHub API adresi oluşturulur.


print("✅ CHECKPOINT 04.5: GitHub aday yolları hazır.")

## Hücre 6 — GitHub RAW URL fonksiyonu

In [ ]:
def build_raw_url(relative_path):
    """Repo içindeki göreli yolu GitHub RAW URL adresine dönüştürür."""

    clean_path = str(relative_path).lstrip("/")
    # Yolun başındaki olası eğik çizgi kaldırılır.

    return (
        f"https://raw.githubusercontent.com/"
        f"{GITHUB_OWNER}/{GITHUB_REPOSITORY}/"
        f"{GITHUB_BRANCH}/{clean_path}"
    )
    # Tam GitHub RAW adresi döndürülür.


print("✅ CHECKPOINT 04.6: RAW URL fonksiyonu hazır.")

## Hücre 7 — GitHub repo ağacını okuma fonksiyonu

In [ ]:
def get_repository_file_paths():
    """GitHub API üzerinden repo içindeki bütün dosya yollarını döndürür."""

    response = requests.get(
        GITHUB_API_TREE_URL,
        timeout=REQUEST_TIMEOUT,
        headers={
            "Accept": "application/vnd.github+json",
        },
    )
    # Recursive GitHub repo ağacı istenir.

    response.raise_for_status()
    # HTTP hata kodlarında exception oluşturulur.

    payload = response.json()
    # JSON yanıtı Python sözlüğüne dönüştürülür.

    file_paths = [
        entry.get("path")
        for entry in payload.get("tree", [])
        if entry.get("type") == "blob" and entry.get("path")
    ]
    # Yalnızca gerçek dosya girişlerinin yolları seçilir.

    if not file_paths:
        # Repo ağacı boşsa kontrol edilir.

        raise RuntimeError(
            "GitHub API repo ağacında dosya döndürmedi."
        )
        # Boş repo ağacıyla devam edilmez.

    return file_paths
    # Repo içindeki bütün dosya yolları döndürülür.


print("✅ CHECKPOINT 04.7: GitHub repo ağacı fonksiyonu hazır.")

## Hücre 8 — Feature store yolunu çözme fonksiyonu

In [ ]:
def resolve_feature_path(repository_files):
    """Repo içinde modellemeye uygun Mordred feature store yolunu seçer."""

    repository_file_set = set(repository_files)
    # Tam yol kontrolünü hızlandırmak için dosya yolları kümeye dönüştürülür.

    for preferred_path in FEATURE_GITHUB_PATHS:
        # Öncelikli feature yolları sırayla kontrol edilir.

        if preferred_path in repository_file_set:
            # Dosya repoda bulunursa seçilir.

            return preferred_path
            # İlk bulunan öncelikli yol döndürülür.

    accepted_names = {
        TARGET_FEATURE_FILENAME,
        REPOSITORY_FEATURE_FILENAME,
    }
    # Desteklenen feature dosya adları tanımlanır.

    exact_matches = [
        path
        for path in repository_files
        if Path(path).name in accepted_names
    ]
    # Desteklenen dosya adları repo içinde recursive aranır.

    if exact_matches:
        # En az bir tam isim eşleşmesi varsa kontrol edilir.

        exact_matches = sorted(
            exact_matches,
            key=lambda path: (
                Path(path).name != TARGET_FEATURE_FILENAME,
                "01_mordred_feature_store" not in path,
                len(Path(path).parts),
                path,
            ),
        )
        # Target dosyası ve gerçek feature-store klasörü önceliklendirilir.

        return exact_matches[0]
        # En uygun eşleşme döndürülür.

    keyword_matches = [
        path
        for path in repository_files
        if (
            str(path).lower().endswith(".csv")
            and "mordred" in Path(path).name.lower()
            and (
                "feature" in Path(path).name.lower()
                or "model_ready" in Path(path).name.lower()
            )
            and "missingness" not in Path(path).name.lower()
            and "index" not in Path(path).name.lower()
            and "prediction" not in Path(path).name.lower()
            and "importance" not in Path(path).name.lower()
            and "metrics" not in Path(path).name.lower()
        )
    ]
    # Dosya adı değişmişse gerçek descriptor matrisi olabilecek CSV'ler aranır.

    if keyword_matches:
        # Uygun anahtar kelime eşleşmesi varsa kontrol edilir.

        keyword_matches = sorted(
            keyword_matches,
            key=lambda path: (
                "01_mordred_feature_store" not in path,
                len(Path(path).parts),
                path,
            ),
        )
        # Feature-store klasörü ve kısa yollar önceliklendirilir.

        return keyword_matches[0]
        # En uygun aday döndürülür.

    raise FileNotFoundError(
        "GitHub reposunda modellemeye uygun Mordred feature store bulunamadı."
    )
    # Uygun feature CSV yoksa açıklayıcı hata oluşturulur.


print("✅ CHECKPOINT 04.8: Feature store çözümleme fonksiyonu hazır.")

## Hücre 9 — Opsiyonel bundle ve model yolu çözme fonksiyonu

In [ ]:
def resolve_optional_path(
    repository_files,
    preferred_paths,
    expected_filename,
):
    """Opsiyonel bir dosyayı öncelikli yollar veya dosya adıyla arar."""

    repository_file_set = set(repository_files)
    # Tam yol kontrolü için dosya yolları kümeye dönüştürülür.

    for preferred_path in preferred_paths:
        # Öncelikli yollar sırayla kontrol edilir.

        if preferred_path in repository_file_set:
            # Dosya repoda bulunursa seçilir.

            return preferred_path
            # İlk bulunan yol döndürülür.

    filename_matches = [
        path
        for path in repository_files
        if Path(path).name == expected_filename
    ]
    # Dosya repo içinde başka bir klasördeyse tam adıyla aranır.

    if filename_matches:
        # En az bir dosya adı eşleşmesi varsa kontrol edilir.

        filename_matches = sorted(
            filename_matches,
            key=lambda path: (
                len(Path(path).parts),
                path,
            ),
        )
        # Daha kısa repo yolu önceliklendirilir.

        return filename_matches[0]
        # En uygun eşleşme döndürülür.

    return None
    # Opsiyonel dosya bulunamazsa None döndürülür.


print("✅ CHECKPOINT 04.9: Opsiyonel dosya çözümleme fonksiyonu hazır.")

## Hücre 10 — GitHub CSV indirme fonksiyonu

In [ ]:
def download_csv(relative_path):
    """GitHub RAW yolundaki CSV'yi pandas DataFrame olarak döndürür."""

    raw_url = build_raw_url(relative_path)
    # Göreli yol tam RAW URL adresine dönüştürülür.

    response = requests.get(
        raw_url,
        timeout=REQUEST_TIMEOUT,
    )
    # GitHub RAW bağlantısına istek gönderilir.

    response.raise_for_status()
    # HTTP hata kodlarında exception oluşturulur.

    dataframe = pd.read_csv(
        BytesIO(response.content),
        low_memory=False,
    )
    # İndirilen içerik pandas DataFrame'e dönüştürülür.

    if dataframe.empty:
        # DataFrame boşsa kontrol edilir.

        raise ValueError(
            f"GitHub CSV boş: {relative_path}"
        )
        # Boş feature store kabul edilmez.

    return dataframe, raw_url
    # DataFrame ve kullanılan RAW URL döndürülür.


print("✅ CHECKPOINT 04.10: GitHub CSV indirme fonksiyonu hazır.")

## Hücre 11 — GitHub JSON indirme fonksiyonu

In [ ]:
def download_json(relative_path):
    """GitHub RAW yolundaki JSON içeriğini sözlük olarak döndürür."""

    raw_url = build_raw_url(relative_path)
    # Göreli yol tam RAW URL adresine dönüştürülür.

    response = requests.get(
        raw_url,
        timeout=REQUEST_TIMEOUT,
    )
    # GitHub RAW bağlantısından JSON dosyası istenir.

    response.raise_for_status()
    # HTTP hata kodlarında exception oluşturulur.

    return response.json(), raw_url
    # JSON sözlüğü ve kullanılan RAW URL döndürülür.


print("✅ CHECKPOINT 04.11: GitHub JSON indirme fonksiyonu hazır.")

## Hücre 12 — GitHub binary dosya indirme fonksiyonu

In [ ]:
def download_binary(relative_path, destination):
    """GitHub RAW binary dosyasını yerel geçici yola indirir."""

    raw_url = build_raw_url(relative_path)
    # Göreli yol tam RAW URL adresine dönüştürülür.

    response = requests.get(
        raw_url,
        timeout=REQUEST_TIMEOUT,
    )
    # Binary dosya GitHub RAW bağlantısından istenir.

    response.raise_for_status()
    # HTTP hata kodlarında exception oluşturulur.

    destination.write_bytes(
        response.content
    )
    # Binary içerik yerel geçici dosyaya yazılır.

    return destination, raw_url
    # Yerel dosya yolu ve kullanılan RAW URL döndürülür.


print("✅ CHECKPOINT 04.12: GitHub binary indirme fonksiyonu hazır.")

## Hücre 13 — GitHub repo ağacının okunması

In [ ]:
repository_files = get_repository_file_paths()
# GitHub reposundaki bütün dosya yolları bir kez okunur.


print("Repo içindeki dosya sayısı:", len(repository_files))
print("✅ CHECKPOINT 04.13: GitHub repo ağacı başarıyla okundu.")

## Hücre 14 — Feature store yolunun seçilmesi

In [ ]:
FEATURE_GITHUB_PATH = resolve_feature_path(
    repository_files
)
# Mevcut repo yapısına uygun feature store yolu otomatik seçilir.


FEATURE_GITHUB_URL = build_raw_url(
    FEATURE_GITHUB_PATH
)
# Seçilen feature store yolu RAW URL adresine dönüştürülür.


print("Seçilen feature store:", FEATURE_GITHUB_PATH)
print("GitHub RAW URL:", FEATURE_GITHUB_URL)
print("✅ CHECKPOINT 04.14: Feature store yolu seçildi.")

## Hücre 15 — Feature store CSV'nin okunması

In [ ]:
df, FEATURE_GITHUB_URL = download_csv(
    FEATURE_GITHUB_PATH
)
# Seçilen Mordred feature store GitHub'dan okunur.


print("Feature store boyutu:", df.shape)
display(df.head(10))
print("✅ CHECKPOINT 04.15: Mordred feature store başarıyla okundu.")

## Hücre 16 — Bundle ve model kaynaklarının aranması

In [ ]:
BUNDLE_GITHUB_PATH = resolve_optional_path(
    repository_files,
    BUNDLE_GITHUB_PATHS,
    BUNDLE_FILENAME,
)
# Model bundle GitHub reposunda aranır.


MODEL_GITHUB_PATH = resolve_optional_path(
    repository_files,
    MODEL_GITHUB_PATHS,
    MODEL_FILENAME,
)
# Joblib model GitHub reposunda aranır.


BUNDLE_DRIVE_PATH = DRIVE_ROOT / BUNDLE_FILENAME
# Model bundle için Google Drive fallback yolu oluşturulur.


MODEL_DRIVE_PATH = DRIVE_ROOT / MODEL_FILENAME
# Joblib model için Google Drive fallback yolu oluşturulur.


IMPUTER_DRIVE_PATH = DRIVE_ROOT / IMPUTER_FILENAME
# Imputer için Google Drive fallback yolu oluşturulur.


print("GitHub bundle:", BUNDLE_GITHUB_PATH)
print("Drive bundle mevcut:", BUNDLE_DRIVE_PATH.exists())
print("GitHub model:", MODEL_GITHUB_PATH)
print("Drive model mevcut:", MODEL_DRIVE_PATH.exists())
print("✅ CHECKPOINT 04.16: Bundle ve model kaynakları arandı.")

## Hücre 17 — Model bundle dosyasının yüklenmesi

In [ ]:
model_bundle = None
# Başlangıçta model bundle bulunmadığı kabul edilir.


BUNDLE_SOURCE = None
# Bundle kaynağı daha sonra raporlanmak üzere boş tanımlanır.


if BUNDLE_GITHUB_PATH is not None:
    # Bundle GitHub reposunda bulunduysa kontrol edilir.

    model_bundle, BUNDLE_SOURCE = download_json(
        BUNDLE_GITHUB_PATH
    )
    # Bundle GitHub RAW bağlantısından okunur.

elif BUNDLE_DRIVE_PATH.exists():
    # GitHub'da bundle yoksa Drive fallback kontrol edilir.

    with open(
        BUNDLE_DRIVE_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        # Drive bundle dosyası okuma modunda açılır.

        model_bundle = json.load(file)
        # JSON içeriği Python sözlüğüne dönüştürülür.

    BUNDLE_SOURCE = str(BUNDLE_DRIVE_PATH)
    # Bundle kaynağı Drive yolu olarak saklanır.


print("Bundle bulundu:", model_bundle is not None)
print("Bundle kaynağı:", BUNDLE_SOURCE)
print("✅ CHECKPOINT 04.17: Bundle yükleme adımı tamamlandı.")

## Hücre 18 — Target ve metadata sütun adayları

In [ ]:
TARGET_COLUMN_CANDIDATES = [
    "Target",
    "target",
    "pStandard",
    "pStandard_mean",
    "pIC50",
    "y",
]
# Desteklenen regression target sütun adları tanımlanır.


METADATA_CANDIDATES = [
    "MoleculeID",
    "SMILES",
    "Target",
    "molecule_id",
    "canonical_smiles",
    "parent_smiles",
    "target",
    "pStandard",
    "pStandard_mean",
    "pStandard_std",
    "target_mean",
    "target_std",
    "source_rows",
    "target_source_column",
    "target_chembl_id",
    "parent_chembl_id",
    "MolWt",
    "MolLogP",
    "HBD",
    "HBA",
]
# Model featureı olarak kullanılmaması gereken açıklayıcı sütunlar tanımlanır.


print("✅ CHECKPOINT 04.18: Target ve metadata adayları hazır.")

## Hücre 19 — Sütun seçme fonksiyonu

In [ ]:
def detect_column(dataframe, candidates, label):
    """Aday sütunlardan DataFrame içinde bulunan ilk sütunu döndürür."""

    for column in candidates:
        # Aday sütunlar sırayla kontrol edilir.

        if column in dataframe.columns:
            # Sütun veri setinde bulunursa seçilir.

            return column
            # İlk uygun sütun döndürülür.

    raise ValueError(
        f"{label} sütunu bulunamadı. Adaylar: {candidates}"
    )
    # Hiçbir aday bulunamazsa açıklayıcı hata oluşturulur.


print("✅ CHECKPOINT 04.19: Sütun seçme fonksiyonu hazır.")

## Hücre 20 — Target, metadata ve feature sütunlarının ayrılması

In [ ]:
target_column = detect_column(
    df,
    TARGET_COLUMN_CANDIDATES,
    "Regression target",
)
# Regression target sütunu otomatik belirlenir.


metadata_columns = [
    column
    for column in METADATA_CANDIDATES
    if column in df.columns
]
# Veri içinde gerçekten bulunan metadata sütunları seçilir.


mordred_prefixed_columns = [
    column
    for column in df.columns
    if str(column).startswith("Mordred_")
]
# Mevcut GitHub formatındaki gerçek Mordred descriptorlar belirlenir.


if mordred_prefixed_columns:
    # Mordred_ önekli featurelar varsa kontrol edilir.

    feature_columns = mordred_prefixed_columns
    # Yalnızca gerçek Mordred descriptor sütunları seçilir.

else:
    # Target'a özel workshop CSV'sinde önek yoksa alternatif kola geçilir.

    feature_columns = [
        column
        for column in df.columns
        if column not in metadata_columns
    ]
    # Metadata dışındaki sütunlar descriptor olarak seçilir.


if not feature_columns:
    # Descriptor sütunu bulunup bulunmadığı kontrol edilir.

    raise ValueError(
        "Application Domain ve SHAP için Mordred feature bulunamadı."
    )
    # Feature yoksa pipeline durdurulur.


print("Target sütunu:", target_column)
print("Metadata sütunu sayısı:", len(metadata_columns))
print("Ham feature sayısı:", len(feature_columns))
print("✅ CHECKPOINT 04.20: Sütun grupları ayrıldı.")

## Hücre 21 — Ham feature matrisi ve target dizisi

In [ ]:
X_raw = (
    df[feature_columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)
# Mordred descriptorlar sayısala dönüştürülür.


X_raw = X_raw.replace(
    [
        np.inf,
        -np.inf,
    ],
    np.nan,
)
# Pozitif ve negatif sonsuz değerler NaN'a çevrilir.


y = pd.to_numeric(
    df[target_column],
    errors="coerce",
)
# Regression target sayısal vektöre dönüştürülür.


print("Ham X boyutu:", X_raw.shape)
print("Ham NaN/sonsuz hücre:", int(X_raw.isna().sum().sum()))
print("✅ CHECKPOINT 04.21: Ham feature matrisi ve target dizisi hazır.")

## Hücre 22 — Geçerli target satırlarının seçilmesi

In [ ]:
valid_target_mask = y.notna()
# Sayısal target değeri bulunan satırlar belirlenir.


removed_target_rows = int(
    (~valid_target_mask).sum()
)
# Eksik target nedeniyle çıkarılacak satır sayısı hesaplanır.


df = df.loc[
    valid_target_mask
].reset_index(drop=True)
# Metadata tablosu geçerli target satırlarıyla sınırlandırılır.


X_raw = X_raw.loc[
    valid_target_mask
].reset_index(drop=True)
# Feature matrisi aynı satır maskesiyle hizalanır.


y = y.loc[
    valid_target_mask
].reset_index(drop=True).astype(float)
# Target vektörü aynı satır maskesiyle hizalanır.


if len(df) < 10:
    # Modelleme için yeterli molekül olup olmadığı kontrol edilir.

    raise ValueError(
        "Application Domain ve SHAP için en az 10 molekül gereklidir."
    )
    # Çok küçük veri setiyle devam edilmez.


print("Eksik target nedeniyle çıkarılan:", removed_target_rows)
print("Geçerli molekül:", len(df))
print("✅ CHECKPOINT 04.22: Geçerli target satırları seçildi.")

## Hücre 23 — Eksik ve sabit descriptorların çıkarılması

In [ ]:
all_missing_columns = X_raw.columns[
    X_raw.isna().all(axis=0)
].tolist()
# Bütün moleküllerde eksik olan descriptorlar belirlenir.


X_filtered = X_raw.drop(
    columns=all_missing_columns
).copy()
# Tamamen boş descriptorlar çıkarılır.


missing_ratio = X_filtered.isna().mean()
# Her descriptor için eksik değer oranı hesaplanır.


high_missing_columns = missing_ratio[
    missing_ratio > MAX_FEATURE_MISSING_RATIO
].index.tolist()
# Eksik oranı yüzde 20'yi aşan descriptorlar belirlenir.


X_filtered = X_filtered.drop(
    columns=high_missing_columns
).copy()
# Yüksek eksik oranlı descriptorlar çıkarılır.


constant_columns = [
    column
    for column in X_filtered.columns
    if X_filtered[column].nunique(dropna=True) <= 1
]
# Eksik olmayan değerleri sabit olan descriptorlar belirlenir.


X_filtered = X_filtered.drop(
    columns=constant_columns
).copy()
# Sabit descriptorlar çıkarılır.


feature_columns = X_filtered.columns.tolist()
# Final feature sırası temizlenmiş matris üzerinden güncellenir.


if X_filtered.empty:
    # Temizlik sonrasında descriptor kalıp kalmadığı kontrol edilir.

    raise ValueError(
        "Feature temizliğinden sonra descriptor kalmadı."
    )
    # Boş feature matrisiyle devam edilmez.


print("Tamamen boş çıkarılan:", len(all_missing_columns))
print("Yüksek eksik oranıyla çıkarılan:", len(high_missing_columns))
print("Sabit çıkarılan:", len(constant_columns))
print("Kalan feature:", len(feature_columns))
print("✅ CHECKPOINT 04.23: Descriptor temizliği tamamlandı.")

## Hücre 24 — Train/test indekslerinin belirlenmesi

In [ ]:
bundle_train_indices = (
    model_bundle.get("train_indices")
    if model_bundle is not None
    else None
)
# Bundle varsa eğitim indeksleri alınır.


bundle_test_indices = (
    model_bundle.get("test_indices")
    if model_bundle is not None
    else None
)
# Bundle varsa test indeksleri alınır.


bundle_indices_are_valid = (
    isinstance(bundle_train_indices, list)
    and isinstance(bundle_test_indices, list)
    and len(bundle_train_indices) > 0
    and len(bundle_test_indices) > 0
    and max(bundle_train_indices + bundle_test_indices) < len(df)
    and set(bundle_train_indices).isdisjoint(
        set(bundle_test_indices)
    )
)
# Bundle indekslerinin güncel veri boyutuyla uyumlu ve çakışmasız olduğu kontrol edilir.


if bundle_indices_are_valid:
    # Geçerli bundle split'i varsa kontrol edilir.

    train_indices = np.asarray(
        bundle_train_indices,
        dtype=int,
    )
    # Bundle eğitim indeksleri NumPy dizisine dönüştürülür.

    test_indices = np.asarray(
        bundle_test_indices,
        dtype=int,
    )
    # Bundle test indeksleri NumPy dizisine dönüştürülür.

    SPLIT_SOURCE = "model_bundle"
    # Split kaynağı bundle olarak kaydedilir.

else:
    # Bundle yoksa veya indeksler uyumsuzsa fallback split oluşturulur.

    train_indices, test_indices = train_test_split(
        np.arange(len(df)),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )
    # Tekrar üretilebilir yüzde 80/20 split oluşturulur.

    SPLIT_SOURCE = "generated"
    # Split kaynağı oluşturulan split olarak kaydedilir.


print("Split kaynağı:", SPLIT_SOURCE)
print("Train:", len(train_indices), "| Test:", len(test_indices))
print("✅ CHECKPOINT 04.24: Train/test indeksleri hazır.")

## Hücre 25 — Train ve test matrislerinin oluşturulması

In [ ]:
X_train_raw = X_filtered.iloc[
    train_indices
].copy()
# Eğitim feature matrisi imputasyon öncesi oluşturulur.


X_test_raw = X_filtered.iloc[
    test_indices
].copy()
# Test feature matrisi imputasyon öncesi oluşturulur.


y_train = y.iloc[
    train_indices
].copy()
# Eğitim target vektörü oluşturulur.


y_test = y.iloc[
    test_indices
].copy()
# Test target vektörü oluşturulur.


print("X_train_raw:", X_train_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("✅ CHECKPOINT 04.25: Train ve test ham matrisleri hazır.")

## Hücre 26 — Train tabanlı median imputasyon

In [ ]:
feature_imputer = SimpleImputer(
    strategy="median",
)
# Eksik descriptorları eğitim medyanıyla dolduracak imputer oluşturulur.


X_train = pd.DataFrame(
    feature_imputer.fit_transform(X_train_raw),
    columns=feature_columns,
    index=X_train_raw.index,
).astype(np.float32)
# Imputer yalnızca eğitim setinde fit edilir ve eğitim matrisi dönüştürülür.


X_test = pd.DataFrame(
    feature_imputer.transform(X_test_raw),
    columns=feature_columns,
    index=X_test_raw.index,
).astype(np.float32)
# Eğitimde öğrenilen medyanlar değiştirilmeden test setine uygulanır.


if not np.isfinite(X_train.to_numpy()).all():
    # Eğitim matrisinde geçersiz değer kalıp kalmadığı kontrol edilir.

    raise AssertionError(
        "İmputasyon sonrasında eğitim matrisinde geçersiz değer kaldı."
    )
    # Geçersiz eğitim matrisi kabul edilmez.


if not np.isfinite(X_test.to_numpy()).all():
    # Test matrisinde geçersiz değer kalıp kalmadığı kontrol edilir.

    raise AssertionError(
        "İmputasyon sonrasında test matrisinde geçersiz değer kaldı."
    )
    # Geçersiz test matrisi kabul edilmez.


print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("✅ CHECKPOINT 04.26: Train tabanlı imputasyon tamamlandı.")

## Hücre 27 — Model üretme fonksiyonu

In [ ]:
MODEL_BUILDERS = {
    "ExtraTreesRegressor": ExtraTreesRegressor,
    "RandomForestRegressor": RandomForestRegressor,
    "GradientBoostingRegressor": GradientBoostingRegressor,
}
# Desteklenen ağaç model adları sklearn sınıflarıyla eşlenir.


DEFAULT_MODEL_NAME = "RandomForestRegressor"
# Bundle veya uyumlu model yoksa kullanılacak fallback model seçilir.


DEFAULT_MODEL_PARAMS = {
    "n_estimators": 500,
    "min_samples_leaf": 2,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}
# Workshop için kararlı ve aşırı uyumu azaltılmış Random Forest parametreleri tanımlanır.


def build_model(model_name, model_params):
    """Model adına göre yeni sklearn ağaç modeli oluşturur."""

    if model_name not in MODEL_BUILDERS:
        # Desteklenmeyen model adı kontrol edilir.

        model_name = DEFAULT_MODEL_NAME
        # Desteklenmeyen ad fallback Random Forest ile değiştirilir.

        model_params = DEFAULT_MODEL_PARAMS.copy()
        # Fallback parametreleri kullanılır.

    model_class = MODEL_BUILDERS[model_name]
    # Model adına karşılık gelen sklearn sınıfı alınır.

    return model_class(
        **model_params
    )
    # Yeni ve fit edilmemiş model nesnesi döndürülür.


print("✅ CHECKPOINT 04.27: Model üretme fonksiyonu hazır.")

## Hücre 28 — Model dosyasının aranması ve yüklenmesi

In [ ]:
loaded_model = None
# Başlangıçta uyumlu model yüklenmediği kabul edilir.


MODEL_SOURCE = None
# Model kaynağı daha sonra raporlanmak üzere boş tanımlanır.


temporary_model_path = Path(
    f"/content/{MODEL_FILENAME}"
)
# GitHub binary model indirilirse kullanılacak geçici yerel yol oluşturulur.


if MODEL_GITHUB_PATH is not None:
    # Model GitHub reposunda bulunduysa kontrol edilir.

    try:
        # Binary indirme veya joblib yükleme hatalarını yakalamak için try kullanılır.

        temporary_model_path, model_url = download_binary(
            MODEL_GITHUB_PATH,
            temporary_model_path,
        )
        # GitHub model dosyası geçici yerel alana indirilir.

        loaded_model = joblib.load(
            temporary_model_path
        )
        # İndirilen joblib model belleğe yüklenir.

        MODEL_SOURCE = model_url
        # Model kaynağı GitHub RAW URL olarak saklanır.

    except Exception as model_error:
        # GitHub model dosyası bozuk veya uyumsuzsa hata yakalanır.

        print("GitHub modeli yüklenemedi:", model_error)
        # Hata bilgi amaçlı gösterilir.

        loaded_model = None
        # Uyumlu model olmadığı belirtilir.

elif MODEL_DRIVE_PATH.exists():
    # GitHub'da model yoksa Drive fallback kontrol edilir.

    try:
        # Drive model yükleme hatalarını yakalamak için try kullanılır.

        loaded_model = joblib.load(
            MODEL_DRIVE_PATH
        )
        # Drive model dosyası belleğe yüklenir.

        MODEL_SOURCE = str(
            MODEL_DRIVE_PATH
        )
        # Model kaynağı Drive yolu olarak saklanır.

    except Exception as model_error:
        # Drive modeli bozuk veya uyumsuzsa hata yakalanır.

        print("Drive modeli yüklenemedi:", model_error)
        # Hata bilgi amaçlı gösterilir.

        loaded_model = None
        # Uyumlu model olmadığı belirtilir.


print("Model dosyası yüklendi:", loaded_model is not None)
print("Model kaynağı:", MODEL_SOURCE)
print("✅ CHECKPOINT 04.28: Model yükleme adımı tamamlandı.")

## Hücre 29 — Yüklenen modelin feature uyumluluğu

In [ ]:
model_is_compatible = False
# Başlangıçta yüklenen model uyumsuz kabul edilir.


if loaded_model is not None:
    # Bir model yüklenmişse feature uyumluluğu kontrol edilir.

    expected_feature_count = getattr(
        loaded_model,
        "n_features_in_",
        None,
    )
    # Modelin eğitimde kullandığı feature sayısı alınır.

    model_is_compatible = (
        expected_feature_count == X_train.shape[1]
    )
    # Model feature sayısı güncel matrisle karşılaştırılır.


print("Yüklenen model feature uyumlu:", model_is_compatible)
print("✅ CHECKPOINT 04.29: Model uyumluluğu kontrol edildi.")

## Hücre 30 — Final modelin hazırlanması

In [ ]:
if model_is_compatible:
    # Yüklenen model güncel feature matrisiyle uyumluysa kontrol edilir.

    final_model = loaded_model
    # Uyumlu model doğrudan kullanılır.

    MODEL_ACTION = "loaded"
    # Modelin yeniden eğitilmediği kaydedilir.

else:
    # Uyumlu kayıtlı model yoksa yeni model oluşturulur.

    selected_model_name = (
        model_bundle.get("selected_model_name")
        if model_bundle is not None
        else DEFAULT_MODEL_NAME
    )
    # Bundle varsa seçilen model adı alınır; yoksa Random Forest kullanılır.

    selected_model_params = (
        model_bundle.get("selected_model_params")
        if model_bundle is not None
        else DEFAULT_MODEL_PARAMS.copy()
    )
    # Bundle parametreleri varsa alınır; yoksa fallback parametreleri kullanılır.

    if selected_model_name == "RandomForestRegressor":
        # Random Forest parametrelerinin Colab ile uyumlu olması sağlanır.

        selected_model_params = {
            **DEFAULT_MODEL_PARAMS,
            **(selected_model_params or {}),
        }
        # Eksik Random Forest parametreleri fallback değerlerle tamamlanır.

    final_model = build_model(
        selected_model_name,
        selected_model_params,
    )
    # Yeni ve temiz ağaç modeli oluşturulur.

    final_model.fit(
        X_train,
        y_train,
    )
    # Model yalnızca eğitim verisinde bir kez fit edilir.

    MODEL_ACTION = "trained"
    # Yeni model eğitildiği kaydedilir.

    MODEL_SOURCE = "generated_from_github_feature_store"
    # Model kaynağı GitHub feature store üzerinden oluşturulan model olarak saklanır.


print("Model işlemi:", MODEL_ACTION)
print("Final model:", final_model.__class__.__name__)
print("✅ CHECKPOINT 04.30: Final model hazır.")

## Hücre 31 — Train ve test tahminlerinin oluşturulması

In [ ]:
train_predictions = final_model.predict(
    X_train
)
# Eğitim seti tahminleri oluşturulur.


test_predictions = final_model.predict(
    X_test
)
# Test seti tahminleri oluşturulur.


train_residuals = (
    y_train.to_numpy()
    - train_predictions
)
# Eğitim artıkları hesaplanır.


test_residuals = (
    y_test.to_numpy()
    - test_predictions
)
# Test artıkları hesaplanır.


print("Train tahmin:", len(train_predictions))
print("Test tahmin:", len(test_predictions))
print("✅ CHECKPOINT 04.31: Tahminler ve artıklar hazır.")

## Hücre 32 — PCA için eğitim tabanlı standardizasyon

In [ ]:
ad_scaler = StandardScaler()
# PCA öncesi featureları standardize edecek scaler oluşturulur.


X_train_scaled = ad_scaler.fit_transform(
    X_train
)
# Scaler yalnızca eğitim setinde fit edilir.


X_test_scaled = ad_scaler.transform(
    X_test
)
# Eğitimde öğrenilen dönüşüm test setine uygulanır.


print("✅ CHECKPOINT 04.32: Eğitim tabanlı standardizasyon tamamlandı.")

## Hücre 33 — PCA bileşen sayısının belirlenmesi

In [ ]:
pca_full = PCA(
    svd_solver="full"
)
# Yüzde 95 varyans için gereken bileşen sayısını belirleyecek PCA oluşturulur.


pca_full.fit(
    X_train_scaled
)
# PCA yalnızca standardize eğitim verisinde fit edilir.


cumulative_variance = np.cumsum(
    pca_full.explained_variance_ratio_
)
# Açıklanan varyans oranlarının kümülatif toplamı hesaplanır.


k_95 = int(
    np.searchsorted(
        cumulative_variance,
        0.95,
    )
    + 1
)
# Yüzde 95 varyansı açıklayan ilk bileşen sayısı belirlenir.


max_k_for_leverage = max(
    1,
    int(np.floor(len(X_train) / 3)) - 1,
)
# h*=3(k+1)/n değerini anlamlı tutmak için bileşen üst sınırı belirlenir.


selected_k = min(
    k_95,
    max_k_for_leverage,
    X_train.shape[1],
    len(X_train) - 1,
)
# PCA bileşen sayısı veri ve leverage sınırlarıyla uyumlu seçilir.


print("Yüzde 95 varyans bileşeni:", k_95)
print("Kullanılan PCA bileşeni:", selected_k)
print("✅ CHECKPOINT 04.33: PCA bileşen sayısı belirlendi.")

## Hücre 34 — PCA dönüşümünün uygulanması

In [ ]:
ad_pca = PCA(
    n_components=selected_k,
    svd_solver="full",
)
# Seçilen bileşen sayısıyla final PCA oluşturulur.


Z_train = ad_pca.fit_transform(
    X_train_scaled
)
# PCA yalnızca eğitim setinde fit edilir ve eğitim verisi dönüştürülür.


Z_test = ad_pca.transform(
    X_test_scaled
)
# Aynı PCA dönüşümü test setine uygulanır.


print(
    "PCA açıklanan varyans:",
    float(ad_pca.explained_variance_ratio_.sum()),
)
print("✅ CHECKPOINT 04.34: PCA dönüşümü tamamlandı.")

## Hücre 35 — Leverage hesaplama fonksiyonu

In [ ]:
def add_intercept(matrix):
    """PCA skor matrisinin başına sabit terim sütunu ekler."""

    return np.column_stack(
        [
            np.ones(matrix.shape[0]),
            matrix,
        ]
    )
    # Sabit terim ve PCA skorları birleştirilerek döndürülür.


def calculate_leverage(
    matrix,
    inverse_cross_product,
):
    """Bir skor matrisi için satır bazlı leverage değerlerini hesaplar."""

    return np.einsum(
        "ij,jk,ik->i",
        matrix,
        inverse_cross_product,
        matrix,
    )
    # Her satır için x_i'(X'X)^-1x_i değeri hesaplanır.


print("✅ CHECKPOINT 04.35: Leverage fonksiyonları hazır.")

## Hücre 36 — Train ve test leverage değerlerinin hesaplanması

In [ ]:
A_train = add_intercept(
    Z_train
)
# Eğitim PCA skorlarına sabit terim eklenir.


A_test = add_intercept(
    Z_test
)
# Test PCA skorlarına sabit terim eklenir.


inverse_cross_product = np.linalg.pinv(
    A_train.T @ A_train
)
# Eğitim tasarım matrisinin çapraz çarpımının pseudo-inverse değeri hesaplanır.


train_leverage = calculate_leverage(
    A_train,
    inverse_cross_product,
)
# Eğitim leverage değerleri hesaplanır.


test_leverage = calculate_leverage(
    A_test,
    inverse_cross_product,
)
# Test leverage değerleri eğitim referans uzayına göre hesaplanır.


h_star = 3 * (
    selected_k + 1
) / len(X_train)
# Kritik leverage eşiği h*=3(k+1)/n formülüyle hesaplanır.


print("h*:", float(h_star))
print("✅ CHECKPOINT 04.36: Leverage değerleri hesaplandı.")

## Hücre 37 — Standartlaştırılmış artıkların hesaplanması

In [ ]:
residual_scale = float(
    np.std(
        train_residuals,
        ddof=1,
    )
)
# Eğitim artıklarının standart sapması hesaplanır.


if not np.isfinite(residual_scale) or residual_scale < 1e-8:
    # Eğitim artık ölçeğinin sıfıra çok yakın veya geçersiz olup olmadığı kontrol edilir.

    residual_scale = float(
        np.sqrt(
            np.mean(
                np.square(train_residuals)
            )
        )
    )
    # Alternatif olarak eğitim RMSE değeri kullanılır.


if not np.isfinite(residual_scale) or residual_scale < 1e-8:
    # Alternatif ölçek de kullanılamazsa kontrol edilir.

    residual_scale = 1.0
    # Sayısal bölme hatasını önlemek için güvenli ölçek kullanılır.


train_standardized_residuals = (
    train_residuals / residual_scale
)
# Eğitim artıkları eğitim artık ölçeğine bölünür.


test_standardized_residuals = (
    test_residuals / residual_scale
)
# Test artıkları aynı eğitim ölçeğine bölünür.


print("Artık ölçeği:", residual_scale)
print("✅ CHECKPOINT 04.37: Standartlaştırılmış artıklar hazır.")

## Hücre 38 — Application Domain etiketlerinin oluşturulması

In [ ]:
train_ad_in = (
    (train_leverage <= h_star)
    & (
        np.abs(
            train_standardized_residuals
        ) <= 3
    )
)
# Eğitim molekülleri leverage ve ±3 artık sınırlarına göre AD-in olarak belirlenir.


test_ad_in = (
    (test_leverage <= h_star)
    & (
        np.abs(
            test_standardized_residuals
        ) <= 3
    )
)
# Test molekülleri aynı kurallarla AD-in olarak belirlenir.


print("Train AD-in:", int(train_ad_in.sum()), "/", len(train_ad_in))
print("Test AD-in:", int(test_ad_in.sum()), "/", len(test_ad_in))
print("✅ CHECKPOINT 04.38: Application Domain etiketleri hazır.")

## Hücre 39 — AD sonuç tablolarının hazırlanması

In [ ]:
train_ad_output = (
    df.iloc[
        train_indices
    ][metadata_columns]
    .copy()
)
# Eğitim satırlarının metadata tablosu oluşturulur.


train_ad_output["actual_target"] = y_train.to_numpy()
# Eğitim gerçek target değerleri eklenir.


train_ad_output["predicted_target"] = train_predictions
# Eğitim tahminleri eklenir.


train_ad_output["leverage"] = train_leverage
# Eğitim leverage değerleri eklenir.


train_ad_output["standardized_residual"] = (
    train_standardized_residuals
)
# Eğitim standartlaştırılmış artıkları eklenir.


train_ad_output["h_star"] = h_star
# Kritik leverage eşiği her satıra eklenir.


train_ad_output["AD_status"] = np.where(
    train_ad_in,
    "AD-in",
    "AD-out",
)
# Eğitim Application Domain etiketi eklenir.


test_ad_output = (
    df.iloc[
        test_indices
    ][metadata_columns]
    .copy()
)
# Test satırlarının metadata tablosu oluşturulur.


test_ad_output["actual_target"] = y_test.to_numpy()
# Test gerçek target değerleri eklenir.


test_ad_output["predicted_target"] = test_predictions
# Test tahminleri eklenir.


test_ad_output["leverage"] = test_leverage
# Test leverage değerleri eklenir.


test_ad_output["standardized_residual"] = (
    test_standardized_residuals
)
# Test standartlaştırılmış artıkları eklenir.


test_ad_output["h_star"] = h_star
# Kritik leverage eşiği her satıra eklenir.


test_ad_output["AD_status"] = np.where(
    test_ad_in,
    "AD-in",
    "AD-out",
)
# Test Application Domain etiketi eklenir.


print("✅ CHECKPOINT 04.39: AD sonuç tabloları hazır.")

## Hücre 40 — AD çıktı yolları

In [ ]:
train_ad_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_train_application_domain.csv"
)
# Eğitim AD CSV yolu oluşturulur.


test_ad_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_test_application_domain.csv"
)
# Test AD CSV yolu oluşturulur.


williams_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_Williams_plot.png"
)
# Williams plot Drive yolu oluşturulur.


train_ad_output.to_csv(
    train_ad_path,
    index=False,
)
# Eğitim AD tablosu Google Drive'a kaydedilir.


test_ad_output.to_csv(
    test_ad_path,
    index=False,
)
# Test AD tablosu Google Drive'a kaydedilir.


print("✅ CHECKPOINT 04.40: AD tabloları Drive'a kaydedildi.")

## Hücre 41 — Williams plot

In [ ]:
plt.figure(
    figsize=(10, 7)
)
# Williams plot için yeni figür oluşturulur.


plt.scatter(
    train_leverage,
    train_standardized_residuals,
    alpha=0.55,
    label="Train",
)
# Eğitim molekülleri leverage-artık uzayında çizilir.


plt.scatter(
    test_leverage,
    test_standardized_residuals,
    alpha=0.80,
    marker="^",
    label="Test",
)
# Test molekülleri farklı işaretle çizilir.


plt.axvline(
    h_star,
    linestyle="--",
    label=f"h*={h_star:.3f}",
)
# Kritik leverage sınırı çizilir.


plt.axhline(
    3,
    linestyle="--",
)
# Üst standartlaştırılmış artık sınırı çizilir.


plt.axhline(
    -3,
    linestyle="--",
)
# Alt standartlaştırılmış artık sınırı çizilir.


plt.xlabel("Leverage")
# X ekseni etiketi eklenir.


plt.ylabel("Standartlaştırılmış artık")
# Y ekseni etiketi eklenir.


plt.title(
    f"{TARGET_ID} — PCA tabanlı Williams Plot"
)
# Grafik başlığı eklenir.


plt.legend()
# Train, test ve h* açıklamaları eklenir.


plt.tight_layout()
# Grafik elemanlarının taşması engellenir.


plt.savefig(
    williams_path,
    dpi=300,
    bbox_inches="tight",
)
# Williams plot Google Drive'a kaydedilir.


plt.show()
# Grafik notebook içinde gösterilir.


print("✅ CHECKPOINT 04.41: Williams plot kaydedildi.")

## Hücre 42 — SHAP örnekleminin hazırlanması

In [ ]:
shap_sample_size = min(
    MAX_SHAP_SAMPLES,
    len(X_test),
)
# SHAP örneklem boyutu test seti ve üst sınır dikkate alınarak belirlenir.


X_shap = X_test.sample(
    n=shap_sample_size,
    random_state=RANDOM_STATE,
)
# Test setinden tekrar üretilebilir SHAP örneklemi seçilir.


print("SHAP örneklem boyutu:", len(X_shap))
print("✅ CHECKPOINT 04.42: SHAP örneklemi hazır.")

## Hücre 43 — TreeExplainer ve SHAP değerleri

In [ ]:
shap_explainer = shap.TreeExplainer(
    final_model
)
# Final ağaç modeli için TreeExplainer oluşturulur.


shap_values = shap_explainer(
    X_shap
)
# Seçilen test örneklemi için SHAP değerleri hesaplanır.


if shap_values.values.ndim == 3:
    # Bazı model/sürüm kombinasyonlarında ek çıktı ekseni varsa kontrol edilir.

    shap_values = shap_values[
        :,
        :,
        0,
    ]
    # Regression için ilk ve tek çıktı ekseni seçilir.


print("SHAP matrisi:", shap_values.values.shape)
print("✅ CHECKPOINT 04.43: SHAP değerleri hesaplandı.")

## Hücre 44 — SHAP önem tablosu

In [ ]:
shap_importance = pd.DataFrame(
    {
        "feature": feature_columns,
        "mean_abs_SHAP": np.abs(
            shap_values.values
        ).mean(axis=0),
    }
)
# Her descriptor için ortalama mutlak SHAP değeri hesaplanır.


shap_importance = shap_importance.sort_values(
    "mean_abs_SHAP",
    ascending=False,
).reset_index(drop=True)
# Featurelar global önem sırasına göre sıralanır.


shap_importance_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_SHAP_feature_importance.csv"
)
# SHAP önem CSV yolu oluşturulur.


shap_importance.to_csv(
    shap_importance_path,
    index=False,
)
# SHAP önem tablosu Google Drive'a kaydedilir.


display(
    shap_importance.head(20)
)
# En önemli ilk 20 descriptor gösterilir.


print("✅ CHECKPOINT 04.44: SHAP önem tablosu kaydedildi.")

## Hücre 45 — SHAP beeswarm grafiği

In [ ]:
shap_beeswarm_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_SHAP_beeswarm.png"
)
# SHAP beeswarm grafik yolu oluşturulur.


shap.plots.beeswarm(
    shap_values,
    max_display=20,
    show=False,
)
# En önemli 20 feature için SHAP beeswarm grafiği oluşturulur.


plt.title(
    f"{TARGET_ID} — SHAP Beeswarm"
)
# Grafik başlığı eklenir.


plt.tight_layout()
# Grafik elemanlarının taşması engellenir.


plt.savefig(
    shap_beeswarm_path,
    dpi=300,
    bbox_inches="tight",
)
# Beeswarm grafiği Google Drive'a kaydedilir.


plt.show()
# Grafik notebook içinde gösterilir.


print("✅ CHECKPOINT 04.45: SHAP beeswarm grafiği kaydedildi.")

## Hücre 46 — SHAP bar grafiği

In [ ]:
shap_bar_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_SHAP_bar.png"
)
# SHAP bar grafik yolu oluşturulur.


shap.plots.bar(
    shap_values,
    max_display=20,
    show=False,
)
# Ortalama mutlak SHAP önemleri bar grafik olarak çizilir.


plt.title(
    f"{TARGET_ID} — Ortalama |SHAP|"
)
# Grafik başlığı eklenir.


plt.tight_layout()
# Grafik elemanlarının taşması engellenir.


plt.savefig(
    shap_bar_path,
    dpi=300,
    bbox_inches="tight",
)
# SHAP bar grafiği Google Drive'a kaydedilir.


plt.show()
# Grafik notebook içinde gösterilir.


print("✅ CHECKPOINT 04.46: SHAP bar grafiği kaydedildi.")

## Hücre 47 — Lokal SHAP örneğinin seçilmesi

In [ ]:
test_absolute_errors = np.abs(
    y_test.to_numpy()
    - test_predictions
)
# Test moleküllerinin mutlak tahmin hataları hesaplanır.


local_position = int(
    np.argmax(
        test_absolute_errors
    )
)
# En yüksek mutlak hataya sahip test örneğinin konumu belirlenir.


X_local = X_test.iloc[
    [
        local_position
    ]
]
# Seçilen tek molekülün feature satırı hazırlanır.


local_shap_values = shap_explainer(
    X_local
)
# Seçilen molekül için lokal SHAP değerleri hesaplanır.


if local_shap_values.values.ndim == 3:
    # Ek çıktı ekseni varsa kontrol edilir.

    local_shap_values = local_shap_values[
        :,
        :,
        0,
    ]
    # Regression için ilk çıktı ekseni seçilir.


print("Lokal SHAP test konumu:", local_position)
print("✅ CHECKPOINT 04.47: Lokal SHAP örneği hazır.")

## Hücre 48 — Lokal SHAP waterfall grafiği

In [ ]:
shap_local_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_SHAP_local_waterfall.png"
)
# Lokal SHAP waterfall grafik yolu oluşturulur.


shap.plots.waterfall(
    local_shap_values[0],
    max_display=15,
    show=False,
)
# Seçilen molekülün en önemli 15 katkısı waterfall grafikte gösterilir.


plt.title(
    f"{TARGET_ID} — Lokal SHAP"
)
# Grafik başlığı eklenir.


plt.tight_layout()
# Grafik elemanlarının taşması engellenir.


plt.savefig(
    shap_local_path,
    dpi=300,
    bbox_inches="tight",
)
# Lokal waterfall grafiği Google Drive'a kaydedilir.


plt.show()
# Grafik notebook içinde gösterilir.


print("✅ CHECKPOINT 04.48: Lokal SHAP grafiği kaydedildi.")

## Hücre 49 — En önemli feature için SHAP dependence

In [ ]:
top_feature = str(
    shap_importance.iloc[0]["feature"]
)
# Ortalama mutlak SHAP değeri en yüksek descriptor seçilir.


shap_dependence_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_SHAP_top_feature_dependence.png"
)
# En önemli feature dependence grafik yolu oluşturulur.


shap.plots.scatter(
    shap_values[
        :,
        top_feature,
    ],
    show=False,
)
# En önemli feature değeri ile SHAP katkısı arasındaki ilişki çizilir.


plt.title(
    f"{TARGET_ID} — SHAP Dependence: {top_feature}"
)
# Grafik başlığı eklenir.


plt.tight_layout()
# Grafik elemanlarının taşması engellenir.


plt.savefig(
    shap_dependence_path,
    dpi=300,
    bbox_inches="tight",
)
# Dependence grafiği Google Drive'a kaydedilir.


plt.show()
# Grafik notebook içinde gösterilir.


print("✅ CHECKPOINT 04.49: SHAP dependence grafiği kaydedildi.")

## Hücre 50 — Final model ve imputerın Drive'a kaydedilmesi

In [ ]:
FINAL_MODEL_PATH = DRIVE_ROOT / MODEL_FILENAME
# Final model için Drive yolu oluşturulur.


FINAL_IMPUTER_PATH = DRIVE_ROOT / IMPUTER_FILENAME
# Final imputer için Drive yolu oluşturulur.


joblib.dump(
    final_model,
    FINAL_MODEL_PATH,
)
# Kullanılan final model Google Drive'a kaydedilir.


joblib.dump(
    feature_imputer,
    FINAL_IMPUTER_PATH,
)
# Eğitim setinde fit edilen median imputer Google Drive'a kaydedilir.


print("✅ CHECKPOINT 04.50: Final model ve imputer Drive'a kaydedildi.")

## Hücre 51 — Güncel model bundle oluşturulması

In [ ]:
updated_bundle = {
    "target_id": TARGET_ID,
    "github_feature_path": FEATURE_GITHUB_PATH,
    "github_feature_url": FEATURE_GITHUB_URL,
    "bundle_source": BUNDLE_SOURCE,
    "model_source": MODEL_SOURCE,
    "model_action": MODEL_ACTION,
    "split_source": SPLIT_SOURCE,
    "target_column": target_column,
    "metadata_columns": metadata_columns,
    "feature_names": feature_columns,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "selected_model_name": final_model.__class__.__name__,
    "model_filename": MODEL_FILENAME,
    "imputer_filename": IMPUTER_FILENAME,
    "train_indices": train_indices.tolist(),
    "test_indices": test_indices.tolist(),
    "dropped_all_missing_features": all_missing_columns,
    "dropped_high_missing_features": high_missing_columns,
    "dropped_constant_features": constant_columns,
}
# Güncel veri, split, model ve feature bilgilerini içeren bundle hazırlanır.


with open(
    BUNDLE_DRIVE_PATH,
    "w",
    encoding="utf-8",
) as file:
    # Bundle dosyası yazma modunda açılır.

    json.dump(
        updated_bundle,
        file,
        ensure_ascii=False,
        indent=2,
    )
    # Güncel bundle Google Drive'a kaydedilir.


print("✅ CHECKPOINT 04.51: Güncel model bundle Drive'a kaydedildi.")

## Hücre 52 — Final AD ve SHAP özetinin hazırlanması

In [ ]:
summary = {
    "target_id": TARGET_ID,
    "github_feature_path": FEATURE_GITHUB_PATH,
    "github_feature_url": FEATURE_GITHUB_URL,
    "model_source": MODEL_SOURCE,
    "model_action": MODEL_ACTION,
    "model_name": final_model.__class__.__name__,
    "split_source": SPLIT_SOURCE,
    "n_molecules": int(len(df)),
    "n_train": int(len(train_indices)),
    "n_test": int(len(test_indices)),
    "n_features": int(len(feature_columns)),
    "pca_components": int(selected_k),
    "pca_explained_variance": float(
        ad_pca.explained_variance_ratio_.sum()
    ),
    "h_star": float(h_star),
    "train_AD_in": int(train_ad_in.sum()),
    "train_AD_out": int((~train_ad_in).sum()),
    "test_AD_in": int(test_ad_in.sum()),
    "test_AD_out": int((~test_ad_in).sum()),
    "top_SHAP_feature": top_feature,
}
# Final analiz özeti sözlük olarak hazırlanır.


summary_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_AD_SHAP_summary.json"
)
# Final özet JSON Drive yolu oluşturulur.


with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    # Final özet dosyası yazma modunda açılır.

    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )
    # Final özet okunabilir JSON biçiminde Drive'a kaydedilir.


display(
    pd.DataFrame(
        {
            "metric": list(summary.keys()),
            "value": list(summary.values()),
        }
    )
)
# Final özet tablo halinde notebook içinde gösterilir.


print("✅ CHECKPOINT 04.52: Final AD ve SHAP özeti kaydedildi.")

## Hücre 53 — Bütün çıktıların son kontrolü

In [ ]:
OUTPUT_PATHS = [
    train_ad_path,
    test_ad_path,
    williams_path,
    shap_importance_path,
    shap_beeswarm_path,
    shap_bar_path,
    shap_local_path,
    shap_dependence_path,
    FINAL_MODEL_PATH,
    FINAL_IMPUTER_PATH,
    BUNDLE_DRIVE_PATH,
    summary_path,
]
# Üretilmesi beklenen bütün Drive çıktı yolları listelenir.


for output_path in OUTPUT_PATHS:
    # Bütün çıktılar sırayla kontrol edilir.

    if not output_path.exists():
        # Dosyanın gerçekten oluşup oluşmadığı kontrol edilir.

        raise IOError(
            f"Drive çıktısı oluşturulamadı: {output_path}"
        )
        # Eksik çıktı varsa pipeline durdurulur.

    if output_path.stat().st_size == 0:
        # Dosyanın boş olup olmadığı kontrol edilir.

        raise IOError(
            f"Drive çıktısı boş oluşturuldu: {output_path}"
        )
        # Boş çıktı kabul edilmez.

    print(
        "[Drive kaydı]",
        output_path,
        f"({output_path.stat().st_size:,} byte)",
    )
    # Başarılı çıktı yolu ve dosya boyutu gösterilir.


print(
    "✅ CHECKPOINT 04.53: Application Domain ve SHAP notebooku tamamlandı."
)

## Notebook sonu

Bu sürümde:

- Sabit ve bulunmayan GitHub feature URL kullanılmaz.
- Mevcut repo içindeki `mordred_2d_features.csv` otomatik bulunur.
- Model bundle GitHub'da yoksa Google Drive kontrol edilir.
- Uyumlu model bulunamazsa model yalnızca bir kez eğitilir.
- `NaN` ve sonsuz Mordred değerleri train tabanlı median imputasyonla işlenir.
- Train/test split yalnızca bir kez oluşturulur.
- PCA, scaler ve imputer yalnızca eğitim setinde fit edilir.
- Bütün sonuçlar Google Drive'a kaydedilir.